In [74]:
import numpy as np
import pandas as pd
import nibabel as nib
import json

from pathlib import Path
from nilearn import datasets
from nilearn.image import resample_to_img
from nilearn.maskers import NiftiLabelsMasker

In [76]:
PROJECT_DIR = Path(
    r"C:\Users\praja\functional-connectivity"
)

PREPROCESSED_FILE = (
    PROJECT_DIR / "milestone_3" / "preprocessed_bold.nii.gz"
)

assert PREPROCESSED_FILE.exists(), "Preprocessed file missing!"

fmri_img = nib.load(str(PREPROCESSED_FILE))

print("Dimensions:", fmri_img.shape)
print("Timepoints:", fmri_img.shape[3])
print("TR:", fmri_img.header.get_zooms()[3])

Dimensions: (104, 104, 72, 488)
Timepoints: 488
TR: 0.8


In [78]:
atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100,
    yeo_networks=7,
    resolution_mm=2
)

atlas_img = nib.load(atlas.maps)

print("Atlas dimensions:", atlas_img.shape)
print("Atlas labels:", len(atlas.labels))

[fetch_atlas_schaefer_2018] Dataset directory found: C:\Users\praja\nilearn_data\schaefer_2018
Atlas dimensions: (91, 109, 91)
Atlas labels: 101


In [80]:
print("fMRI orientation:",
      nib.aff2axcodes(fmri_img.affine))

print("Atlas orientation:",
      nib.aff2axcodes(atlas_img.affine))

print("fMRI affine:\n", fmri_img.affine)
print("Atlas affine:\n", atlas_img.affine)

fMRI orientation: ('L', 'A', 'S')
Atlas orientation: ('L', 'A', 'S')
fMRI affine:
 [[-1.99630272e+00  7.38988444e-02 -9.65107232e-02  1.02097656e+02]
 [ 8.22728723e-02  1.99038637e+00 -1.77745119e-01 -7.22163391e+01]
 [-8.94792303e-02  1.81386650e-01  1.98974681e+00 -6.34207611e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
Atlas affine:
 [[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]


In [82]:
resampled_atlas = resample_to_img(
    atlas_img,
    fmri_img,
    interpolation="nearest"
)

resampled_data = resampled_atlas.get_fdata()

print("Resampled atlas shape:", resampled_data.shape)
print("fMRI spatial shape:", fmri_img.shape[:3])

Resampled atlas shape: (104, 104, 72)
fMRI spatial shape: (104, 104, 72)


In [84]:
present_labels = np.unique(
    resampled_data.astype(int)
)

present_labels = present_labels[present_labels > 0]

missing_rois = sorted(
    set(range(1, 101)) - set(present_labels.tolist())
)

print("Available ROIs:", len(present_labels))
print("Missing ROIs:", missing_rois)
print("Missing count:", len(missing_rois))

Available ROIs: 97
Missing ROIs: [4, 5, 54]
Missing count: 3


In [86]:
masker = NiftiLabelsMasker(
    labels_img=resampled_atlas,
    standardize=False,
    detrend=False,
    strategy="mean"
)

regional_signals = masker.fit_transform(fmri_img)

print("Regional signals shape:", regional_signals.shape)

C:\Users\praja\AppData\Local\Temp\ipykernel_20392\4219150245.py:8: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  regional_signals = masker.fit_transform(fmri_img)


Regional signals shape: (488, 97)


C:\Users\praja\AppData\Local\Temp\ipykernel_20392\4219150245.py:8: FutureWarning: boolean values for 'standardize' will be deprecated in nilearn 0.15.0.
Use 'zscore_sample' instead of 'True' or use 'None' instead of 'False'.
  regional_signals = masker.fit_transform(fmri_img)


In [88]:
print("NaN values:", np.isnan(regional_signals).sum())
print("Infinite values:", np.isinf(regional_signals).sum())

print("All finite:", np.isfinite(regional_signals).all())

print("Signal mean:", np.mean(regional_signals))
print("Signal standard deviation:", np.std(regional_signals))

NaN values: 0
Infinite values: 0
All finite: True
Signal mean: -0.0032126148156455033
Signal standard deviation: 0.07846857772423842


In [98]:
print("Signal columns:", regional_signals.shape[1])
print("Masker labels:", len(masker.labels_))
print("Region IDs:", len(masker.region_ids_))

print("\nRegion IDs:")
print(masker.region_ids_)

# Get non-background atlas labels
atlas_label_values = np.unique(
    resampled_data.astype(int)
)

atlas_label_values = sorted(
    atlas_label_values[atlas_label_values > 0]
)

# Map atlas label IDs to their names
label_names = {
    i: (
        label.decode("utf-8")
        if isinstance(label, bytes)
        else str(label)
    )
    for i, label in enumerate(atlas.labels)
}

# Retain only labels represented in the extracted signals
roi_names = [
    label_names[label_id]
    for label_id in atlas_label_values
    if label_id in label_names
]

print("Signal columns:", regional_signals.shape[1])
print("Matched ROI names:", len(roi_names))

Signal columns: 97
Masker labels: 98
Region IDs: 98

Region IDs:
{'background': 0, 0: 1.0, 1: 2.0, 2: 3.0, 3: 6.0, 4: 7.0, 5: 8.0, 6: 9.0, 7: 10.0, 8: 11.0, 9: 12.0, 10: 13.0, 11: 14.0, 12: 15.0, 13: 16.0, 14: 17.0, 15: 18.0, 16: 19.0, 17: 20.0, 18: 21.0, 19: 22.0, 20: 23.0, 21: 24.0, 22: 25.0, 23: 26.0, 24: 27.0, 25: 28.0, 26: 29.0, 27: 30.0, 28: 31.0, 29: 32.0, 30: 33.0, 31: 34.0, 32: 35.0, 33: 36.0, 34: 37.0, 35: 38.0, 36: 39.0, 37: 40.0, 38: 41.0, 39: 42.0, 40: 43.0, 41: 44.0, 42: 45.0, 43: 46.0, 44: 47.0, 45: 48.0, 46: 49.0, 47: 50.0, 48: 51.0, 49: 52.0, 50: 53.0, 51: 55.0, 52: 56.0, 53: 57.0, 54: 58.0, 55: 59.0, 56: 60.0, 57: 61.0, 58: 62.0, 59: 63.0, 60: 64.0, 61: 65.0, 62: 66.0, 63: 67.0, 64: 68.0, 65: 69.0, 66: 70.0, 67: 71.0, 68: 72.0, 69: 73.0, 70: 74.0, 71: 75.0, 72: 76.0, 73: 77.0, 74: 78.0, 75: 79.0, 76: 80.0, 77: 81.0, 78: 82.0, 79: 83.0, 80: 84.0, 81: 85.0, 82: 86.0, 83: 87.0, 84: 88.0, 85: 89.0, 86: 90.0, 87: 91.0, 88: 92.0, 89: 93.0, 90: 94.0, 91: 95.0, 92: 96.0, 93: 

In [100]:
OUTPUT_DIR = PROJECT_DIR / "milestone_4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert len(roi_names) == regional_signals.shape[1], (
    "ROI mapping still does not match. Stop and inspect."
)

df = pd.DataFrame(
    regional_signals,
    columns=roi_names
)

csv_path = OUTPUT_DIR / "regional_time_series.csv"

df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print("Data dimensions:", df.shape)

Saved: C:\Users\praja\functional-connectivity\milestone_4\regional_time_series.csv
Data dimensions: (488, 97)


In [102]:
metadata = {
    "dataset": "ds005237",
    "participant": "sub-NDARINVYT858CBN",
    "atlas": "Schaefer 2018",
    "requested_regions": 100,
    "extracted_regions": int(regional_signals.shape[1]),
    "timepoints": int(regional_signals.shape[0]),
    "signal_shape": list(regional_signals.shape),
    "nan_values": int(np.isnan(regional_signals).sum()),
    "infinite_values": int(np.isinf(regional_signals).sum()),
    "status": "regional time-series extraction completed"
}

metadata_path = OUTPUT_DIR / "milestone_4_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("MILESTONE 4 OUTPUTS SAVED")
print(json.dumps(metadata, indent=4))

MILESTONE 4 OUTPUTS SAVED
{
    "dataset": "ds005237",
    "participant": "sub-NDARINVYT858CBN",
    "atlas": "Schaefer 2018",
    "requested_regions": 100,
    "extracted_regions": 97,
    "timepoints": 488,
    "signal_shape": [
        488,
        97
    ],
    "nan_values": 0,
    "infinite_values": 0,
    "status": "regional time-series extraction completed"
}


In [104]:
print("CSV exists:", csv_path.exists())
print("Metadata exists:", metadata_path.exists())

saved_df = pd.read_csv(csv_path)

print("Saved dataframe shape:", saved_df.shape)
print("All values finite:", np.isfinite(saved_df.values).all())

CSV exists: True
Metadata exists: True
Saved dataframe shape: (488, 97)
All values finite: True
